# LeetCode 2293 — Min Max Game bằng Python

Link đề: https://leetcode.com/problems/min-max-game/ · Độ khó: **Easy** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minMaxGame(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài **mô phỏng (simulation)**: làm đúng theo luật đề, mỗi vòng thu nhỏ mảng một nửa cho tới khi còn một phần tử.
2. Biết xây mảng mới bằng **list comprehension**, dùng `min`, `max`, kiểm tra chẵn/lẻ bằng `i % 2`, và thao tác **tại chỗ (in-place)** trên mảng.
3. Biết tính **độ dài mảng sau mỗi vòng**: `n -> n/2 -> n/4 -> ... -> 1` (chia đôi `log2(n)` lần) và vì sao tổng công việc là `O(n)`.
4. Viết 5 cách: mảng mới mỗi vòng, ghi đè in-place, đệ quy, `zip`/lát cắt bước nhảy, và dùng cây nhị phân (xếp hạng theo tầng) / viết lại bằng chỉ số.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: đảo `min`/`max`, dùng chỉ số của mảng cũ thay vì mới để quyết định `min`/`max`, sửa mảng khi vẫn đang đọc.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài `n` là **lũy thừa của 2** (`1, 2, 4, 8, ...`). Áp dụng thuật toán sau lên `nums`:

1. Gọi `n` là độ dài hiện tại. Nếu `n == 1` thì **dừng**; kết quả là `nums[0]`.
2. Ngược lại tạo mảng mới `newNums` có độ dài `n / 2`. Với mỗi chỉ số **chẵn** `i` thỏa `0 <= i < n / 2`: `newNums[i] = min(nums[2 * i], nums[2 * i + 1])`. Với mỗi chỉ số **lẻ** `i`: `newNums[i] = max(nums[2 * i], nums[2 * i + 1])`.
3. Thay `nums` bằng `newNums`, quay lại bước 1.

Trả về số cuối cùng còn lại trong `nums`.

| Ví dụ | nums | Output | Các vòng |
|---|---|---|---|
| 1 | `[1,3,5,2,4,8,2,2]` | `1` | `[1,5,4,2]` -> `[1,4]` -> `[1]` |
| 2 | `[3]` | `3` | không cần vòng nào |

Constraints: `1 <= nums.length <= 1024`; `1 <= nums[i] <= 10^9`; `nums.length` là lũy thừa của `2`.

In [1]:
examples = [([1, 3, 5, 2, 4, 8, 2, 2], 1), ([3], 3)]
for nums, expected in examples:
    cur = list(nums)
    rounds = [cur]
    while len(cur) > 1:
        cur = [min(cur[2 * i], cur[2 * i + 1]) if i % 2 == 0 else max(cur[2 * i], cur[2 * i + 1])
               for i in range(len(cur) // 2)]
        rounds.append(cur)
    print(f"nums={nums}: các vòng={rounds} -> kết quả {cur[0]} (mong đợi {expected})")

nums=[1, 3, 5, 2, 4, 8, 2, 2]: các vòng=[[1, 3, 5, 2, 4, 8, 2, 2], [1, 5, 4, 2], [1, 4], [1]] -> kết quả 1 (mong đợi 1)
nums=[3]: các vòng=[[3]] -> kết quả 3 (mong đợi 3)


## 3. Kiến thức nền

### 3.1. Lũy thừa của 2 và số vòng

Độ dài là `n = 2^k`. Mỗi vòng chia đôi: `2^k -> 2^(k-1) -> ... -> 1` nên có đúng `k = log2(n)` vòng. Với `n = 1024` chỉ cần 10 vòng. Tổng số phần tử phải xử lý: `n/2 + n/4 + ... + 1 = n - 1` — tuyến tính, không phải `n log n`.

In [2]:
n = 8
lengths = []
while n >= 1:
    lengths.append(n)
    n //= 2
print(lengths, "tổng các độ dài mới (trừ n):", sum(lengths[1:]))
print((8).bit_length() - 1, "= log2(8)")      # bit_length()-1 cho k khi n = 2**k

[8, 4, 2, 1] tổng các độ dài mới (trừ n): 7
3 = log2(8)


### 3.2. `min`, `max` cho hai số và chẵn/lẻ

`min(a, b)` / `max(a, b)` chọn số nhỏ/lớn hơn. Chỉ số chẵn/lẻ kiểm tra bằng `i % 2 == 0` (chẵn) hoặc `i & 1` (lẻ).

In [3]:
print(min(5, 2), max(5, 2))
print([i % 2 == 0 for i in range(6)])
print([i & 1 for i in range(6)])               # 0 cho chẵn, 1 cho lẻ

2 5
[True, False, True, False, True, False]
[0, 1, 0, 1, 0, 1]


### 3.3. List comprehension có điều kiện kiểu `A if cond else B`

`[x if cond else y for i in range(...)]` tạo list mới. Dùng để viết một vòng `newNums` trong một dòng: chẵn thì `min`, lẻ thì `max`.

In [4]:
nums = [1, 3, 5, 2, 4, 8, 2, 2]
new = [min(nums[2*i], nums[2*i+1]) if i % 2 == 0 else max(nums[2*i], nums[2*i+1])
       for i in range(len(nums) // 2)]
print(new)

[1, 5, 4, 2]


### 3.4. Lát cắt bước nhảy `nums[0::2]`, `nums[1::2]` và `zip`

`nums[0::2]` lấy phần tử chỉ số chẵn, `nums[1::2]` lấy phần tử chỉ số lẻ. `zip(nums[0::2], nums[1::2])` ghép từng **cặp liền nhau** `(nums[0], nums[1])`, `(nums[2], nums[3])`, ... — chính là các cặp mà đề cần.

In [5]:
nums = [1, 3, 5, 2, 4, 8, 2, 2]
print(nums[0::2], nums[1::2])
print(list(zip(nums[0::2], nums[1::2])))

[1, 5, 4, 2] [3, 2, 8, 2]
[(1, 3), (5, 2), (4, 8), (2, 2)]


### 3.5. Ghi đè tại chỗ (in-place) và vì sao an toàn

Vòng này ghi vào `nums[i]` các giá trị dùng `nums[2i]`, `nums[2i+1]`. Vì `2i >= i` (và `2i+1 > i` khi `i >= 0`) nên khi ghi `nums[i]` ta đã đọc xong các ô `2i`, `2i+1` của chính bước này, **và** chưa ghi đè ô nào mà bước sau còn cần (bước sau `i' > i` cần ô `2i' >= 2i+2 > i`). Vì thế ghi đè từ `i = 0` tăng dần là an toàn.

In [6]:
nums = [1, 3, 5, 2, 4, 8, 2, 2]
n = len(nums)
while n > 1:
    n //= 2
    for i in range(n):
        a, b = nums[2 * i], nums[2 * i + 1]
        nums[i] = min(a, b) if i % 2 == 0 else max(a, b)
    print(f"n={n}, nums[:n]={nums[:n]}")

n=4, nums[:n]=[1, 5, 4, 2]
n=2, nums[:n]=[1, 4]
n=1, nums[:n]=[1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung các số đứng thành hàng. Mỗi vòng, ta gom **từng cặp hàng xóm** thành một phần tử mới: cặp thứ nhất lấy số **nhỏ**, cặp thứ hai lấy số **lớn**, cặp thứ ba lại lấy **nhỏ**, ... luân phiên.

Tính tay `[1,3,5,2,4,8,2,2]`:

- Cặp `(1,3)` ở vị trí `0` (chẵn): nhỏ nhất = `1`.
- Cặp `(5,2)` ở vị trí `1` (lẻ): lớn nhất = `5`.
- Cặp `(4,8)` ở vị trí `2` (chẵn): nhỏ nhất = `4`.
- Cặp `(2,2)` ở vị trí `3` (lẻ): lớn nhất = `2`.
- Mảng mới `[1,5,4,2]`. Lặp: `(1,5)`->`1`, `(4,2)`->`4`; mảng `[1,4]`; `(1,4)` ở vị trí `0` -> `1`.

Quan trọng: quy tắc `min`/`max` dựa vào **chỉ số của phần tử trong mảng mới** (`i` chẵn hay lẻ), không phụ thuộc giá trị.

In [7]:
nums = [1, 3, 5, 2, 4, 8, 2, 2]
pairs = [(nums[2*i], nums[2*i+1]) for i in range(len(nums) // 2)]
print(pairs)
print([(p, "min" if i % 2 == 0 else "max") for i, p in enumerate(pairs)])

[(1, 3), (5, 2), (4, 8), (2, 2)]
[((1, 3), 'min'), ((5, 2), 'max'), ((4, 8), 'min'), ((2, 2), 'max')]


## 5. Cách 1 — Brute Force / baseline: tạo mảng mới mỗi vòng

**Ý tưởng:** làm đúng từng chữ của đề: `while len(nums) > 1:` tạo `newNums` rồi gán lại `nums = newNums`.

**Độ phức tạp:** vòng đầu `n/2` phép, vòng sau `n/4`, ... tổng `O(n)`. Bộ nhớ: mỗi vòng cấp phát mảng mới nhưng mảng cũ được thu hồi; đỉnh là `O(n)`. **Nút thắt** (nhỏ): tốn bộ nhớ tạm, có thể ghi đè trực tiếp.

In [8]:
from typing import List

class SolutionNewArray:
    def minMaxGame(self, nums: List[int]) -> int:
        while len(nums) > 1:
            new_nums = [0] * (len(nums) // 2)
            for i in range(len(new_nums)):
                if i % 2 == 0:
                    new_nums[i] = min(nums[2 * i], nums[2 * i + 1])
                else:
                    new_nums[i] = max(nums[2 * i], nums[2 * i + 1])
            nums = new_nums                        # gán tên mới, KHÔNG sửa danh sách của người gọi
        return nums[0]

for arr in ([1, 3, 5, 2, 4, 8, 2, 2], [3]):
    print(arr, "->", SolutionNewArray().minMaxGame(arr))

[1, 3, 5, 2, 4, 8, 2, 2] -> 1
[3] -> 3


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** ghi đè in-place an toàn (mục 3.5), nên không cần mảng phụ: dùng một biến `n` là độ dài hiện tại, mỗi vòng `n //= 2`.

**Insight 2:** bài có **cấu trúc cây nhị phân**: các lá là `nums`, mỗi tầng trên nhìn các cặp liền kề, kết quả là gốc cây. Do đó ta có thể viết **đệ quy** hoặc tính theo tầng.

**Invariant:** sau mỗi vòng, `nums[:n]` chính là mảng `newNums` của đề (với `n` mới).

**Quy tắc:** `n > 1` thì với `i = 0..n/2-1`: chẵn `i` -> `min`, lẻ `i` -> `max`; rồi `n //= 2`.

## 7. Các cách tiếp theo

### Cách 2 — Ghi đè in-place (`O(1)` bộ nhớ phụ)

Dùng chính mảng đầu vào (ta làm trên bản sao để không đổi dữ liệu người gọi trong demo).

In [9]:
from typing import List

class SolutionInPlace:
    def minMaxGame(self, nums: List[int]) -> int:
        nums = list(nums)                          # bản sao cho demo; LeetCode cho phép sửa trực tiếp
        n = len(nums)
        while n > 1:
            n //= 2
            for i in range(n):
                a, b = nums[2 * i], nums[2 * i + 1]
                nums[i] = min(a, b) if i % 2 == 0 else max(a, b)
        return nums[0]

for arr in ([1, 3, 5, 2, 4, 8, 2, 2], [9, 1], [7]):
    print(arr, "->", SolutionInPlace().minMaxGame(arr))

[1, 3, 5, 2, 4, 8, 2, 2] -> 1
[9, 1] -> 1
[7] -> 7


**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(1)` phụ (nếu sửa thẳng đầu vào).

### Cách 3 — Đệ quy

Hàm `play(nums)` trả `nums[0]` nếu `len == 1`, ngược lại tạo mảng nửa rồi gọi `play` lên mảng đó. Độ sâu đệ quy chỉ `log2(n) <= 10`, rất an toàn.

In [10]:
from typing import List

class SolutionRecursive:
    def minMaxGame(self, nums: List[int]) -> int:
        if len(nums) == 1:
            return nums[0]
        half = [min(nums[2*i], nums[2*i+1]) if i % 2 == 0 else max(nums[2*i], nums[2*i+1])
                for i in range(len(nums) // 2)]
        return self.minMaxGame(half)

for arr in ([1, 3, 5, 2, 4, 8, 2, 2], [4, 2], [5]):
    print(arr, "->", SolutionRecursive().minMaxGame(arr))

[1, 3, 5, 2, 4, 8, 2, 2] -> 1
[4, 2] -> 2
[5] -> 5


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)` cho mảng tạm đỉnh ở tầng đầu cộng `O(log n)` ngăn xếp đệ quy.

### Cách 4 — `zip` các cặp + `enumerate`

Ghép các cặp liền kề bằng `zip(nums[0::2], nums[1::2])`, rồi chọn `min`/`max` theo chỉ số trong `enumerate`. Đọc lên gần như đọc đề.

In [11]:
from typing import List

class SolutionZip:
    def minMaxGame(self, nums: List[int]) -> int:
        while len(nums) > 1:
            nums = [min(a, b) if i % 2 == 0 else max(a, b)
                    for i, (a, b) in enumerate(zip(nums[0::2], nums[1::2]))]
        return nums[0]

for arr in ([1, 3, 5, 2, 4, 8, 2, 2], [6, 6], [8]):
    print(arr, "->", SolutionZip().minMaxGame(arr))

[1, 3, 5, 2, 4, 8, 2, 2] -> 1
[6, 6] -> 6
[8] -> 8


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` (lát cắt sao chép).

### Cách 5 — Lặp theo "bước" và tầng (không tạo list)

Thay vì thu nhỏ mảng, giữ nguyên mảng và dùng khoảng cách `step = 1, 2, 4, ...`. Ở mức `step`, phần tử đại diện của khối nằm ở chỉ số `j * 2 * step`. Phần tử thứ `t` trong tầng mới (`t = j`) quyết định `min`/`max` theo `t % 2`. Ta ghi vào **ô đầu khối** — không cần ô phụ.

In [12]:
from typing import List

class SolutionStride:
    def minMaxGame(self, nums: List[int]) -> int:
        a = list(nums)
        n, step = len(a), 1
        while step < n:
            for t, pos in enumerate(range(0, n, 2 * step)):     # t = chỉ số trong tầng mới
                x, y = a[pos], a[pos + step]
                a[pos] = min(x, y) if t % 2 == 0 else max(x, y)
            step *= 2
        return a[0]

for arr in ([1, 3, 5, 2, 4, 8, 2, 2], [2, 9, 4, 4], [1]):
    print(arr, "->", SolutionStride().minMaxGame(arr))

[1, 3, 5, 2, 4, 8, 2, 2] -> 1
[2, 9, 4, 4] -> 2
[1] -> 1


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(1)` phụ. Mỗi tầng ghi vào ô đầu của khối `2*step`, giữ nguyên phần còn lại; đây là cách nhìn "cây nhị phân lưu trong mảng".

## 8. TRACE từng bước

Trace Cách 2 với `nums = [1,3,5,2,4,8,2,2]`:

| Vòng | `n` mới | `i` | cặp `(nums[2i], nums[2i+1])` | `min`/`max` | `nums[:n]` sau khi ghi |
|---|---|---|---|---|---|
| 1 | 4 | 0 | `(1,3)` | min -> 1 | `[1,...]` |
| 1 | 4 | 1 | `(5,2)` | max -> 5 | `[1,5,...]` |
| 1 | 4 | 2 | `(4,8)` | min -> 4 | `[1,5,4,...]` |
| 1 | 4 | 3 | `(2,2)` | max -> 2 | `[1,5,4,2]` |
| 2 | 2 | 0 | `(1,5)` | min -> 1 | `[1,...]` |
| 2 | 2 | 1 | `(4,2)` | max -> 4 | `[1,4]` |
| 3 | 1 | 0 | `(1,4)` | min -> 1 | `[1]` |

Kết quả `1`.

In [13]:
def trace_inplace(nums):
    nums = list(nums)
    n = len(nums)
    vong = 0
    while n > 1:
        n //= 2
        vong += 1
        for i in range(n):
            a, b = nums[2 * i], nums[2 * i + 1]
            op = "min" if i % 2 == 0 else "max"
            nums[i] = min(a, b) if op == "min" else max(a, b)
            print(f"  vòng {vong}, i={i}: ({a},{b}) {op} -> {nums[i]}   nums[:{n}]={nums[:n]}")
    return nums[0]

print("kết quả:", trace_inplace([1, 3, 5, 2, 4, 8, 2, 2]))

  vòng 1, i=0: (1,3) min -> 1   nums[:4]=[1, 3, 5, 2]
  vòng 1, i=1: (5,2) max -> 5   nums[:4]=[1, 5, 5, 2]
  vòng 1, i=2: (4,8) min -> 4   nums[:4]=[1, 5, 4, 2]
  vòng 1, i=3: (2,2) max -> 2   nums[:4]=[1, 5, 4, 2]
  vòng 2, i=0: (1,5) min -> 1   nums[:2]=[1, 5]
  vòng 2, i=1: (4,2) max -> 4   nums[:2]=[1, 4]
  vòng 3, i=0: (1,4) min -> 1   nums[:1]=[1]
kết quả: 1


In [14]:
def trace_stride(nums):
    a = list(nums); n = len(a); step = 1
    while step < n:
        row = []
        for t, pos in enumerate(range(0, n, 2 * step)):
            x, y = a[pos], a[pos + step]
            a[pos] = min(x, y) if t % 2 == 0 else max(x, y)
            row.append((pos, pos + step, a[pos]))
        print(f"  step={step}: (pos, pos+step, giá trị mới)={row}")
        step *= 2
    return a[0]

print("kết quả:", trace_stride([1, 3, 5, 2, 4, 8, 2, 2]))

  step=1: (pos, pos+step, giá trị mới)=[(0, 1, 1), (2, 3, 5), (4, 5, 4), (6, 7, 2)]
  step=2: (pos, pos+step, giá trị mới)=[(0, 2, 1), (4, 6, 4)]
  step=4: (pos, pos+step, giá trị mới)=[(0, 4, 1)]
kết quả: 1


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def minMaxGame(self, nums: List[int]) -> int:
        n = len(nums)
        while n > 1:
            n //= 2
            for i in range(n):
                a, b = nums[2 * i], nums[2 * i + 1]
                nums[i] = min(a, b) if i % 2 == 0 else max(a, b)
        return nums[0]

print(Solution().minMaxGame([1, 3, 5, 2, 4, 8, 2, 2]))
print(Solution().minMaxGame([3]))

1
3


## 10. Vì sao thuật toán đúng

- **Mô phỏng đúng định nghĩa.** Mỗi vòng ta tính đúng `newNums[i]` theo công thức đề, chỉ khác là ghi vào chính mảng.
- **Ghi đè không phá dữ liệu cần dùng.** Khi tính `nums[i]` ta đọc `nums[2i]`, `nums[2i+1]` rồi ghi `nums[i]`. Các phần tử còn cần đọc ở bước `i' > i` nằm ở `2i' >= 2i + 2`, luôn lớn hơn `i`, nên chưa bị ghi đè. Với `i = 0`: ta đọc ô `0,1` rồi ghi ô `0` — đọc trước ghi sau, an toàn.
- **Dừng đúng lúc.** Độ dài giảm một nửa mỗi vòng và là lũy thừa của 2 nên luôn chẵn khi `> 1`; kết thúc khi `n == 1`, đáp án là `nums[0]`.
- **Số vòng hữu hạn.** `log2(n)` vòng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (lũy thừa của 2).

| Cách | Time | Space |
|---|---|---|
| 1. Mảng mới mỗi vòng | `O(n)` | `O(n)` |
| 2. In-place | `O(n)` | `O(1)` phụ |
| 3. Đệ quy | `O(n)` | `O(n)` mảng tầng đầu, `O(log n)` ngăn xếp |
| 4. `zip` | `O(n)` | `O(n)` |
| 5. Stride | `O(n)` | `O(1)` phụ (trên bản sao) |

Tổng công việc: `n/2 + n/4 + ... + 1 < n`, hàm là chuỗi hình học nên **tuyến tính**, không phải `O(n log n)`. Vì `n <= 1024` nên mọi cách đều rất nhanh.

## 12. Bộ test

Đối chiếu tất cả các cách trên nhiều input.

In [16]:
import random

solutions = [SolutionNewArray(), SolutionInPlace(), SolutionRecursive(), SolutionZip(), SolutionStride(), Solution()]

fixed = [
    ([1, 3, 5, 2, 4, 8, 2, 2], 1),
    ([3], 3),
    ([5, 5], 5),
    ([2, 9], 2),             # một vòng, vị trí 0 là min
    ([9, 2], 2),
    ([1, 2, 3, 4], 1),       # [1,4] -> min=1
    ([4, 3, 2, 1], 2),       # [min(4,3), max(2,1)] = [3,2] -> min = 2
    ([7, 7, 7, 7, 7, 7, 7, 7], 7),
]
for nums, expected in fixed:
    for s in solutions:
        assert s.minMaxGame(list(nums)) == expected, (type(s).__name__, nums, expected)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 8 test cố định cho 6 cách


In [17]:
random.seed(2293)
for _ in range(500):
    k = random.randint(0, 10)                      # n = 2**k, tối đa 1024
    n = 2 ** k
    nums = [random.randint(1, random.choice([3, 20, 10**9])) for _ in range(n)]
    ref = SolutionNewArray().minMaxGame(list(nums))
    for s in solutions[1:]:
        assert s.minMaxGame(list(nums)) == ref, (type(s).__name__, nums[:8])
print("500 test ngẫu nhiên đối chiếu: OK (độ dài đến 1024)")

500 test ngẫu nhiên đối chiếu: OK (độ dài đến 1024)


In [18]:
# SolutionNewArray, SolutionRecursive không đổi mảng gốc; cách khác chạy trên bản sao
data = [1, 3, 5, 2, 4, 8, 2, 2]
copy = list(data)
for s in solutions[:5]:
    s.minMaxGame(data)
    assert data == copy, type(s).__name__
print("mảng đầu vào không bị sửa (trừ Solution nộp bài): OK")

mảng đầu vào không bị sửa (trừ Solution nộp bài): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đảo `min` và `max`

Chỉ số **chẵn -> min**, **lẻ -> max**. Đảo ngược cho ra đáp án khác với ví dụ (`[1,3,5,2,4,8,2,2]` sẽ không ra `1`).

In [19]:
def sai_dao(nums):
    nums = list(nums); n = len(nums)
    while n > 1:
        n //= 2
        for i in range(n):
            a, b = nums[2*i], nums[2*i+1]
            nums[i] = max(a, b) if i % 2 == 0 else min(a, b)      # SAI: đảo
    return nums[0]

print("sai :", sai_dao([1, 3, 5, 2, 4, 8, 2, 2]), "| đúng:", SolutionInPlace().minMaxGame([1, 3, 5, 2, 4, 8, 2, 2]))

sai : 3 | đúng: 1


### Lỗi 2: Quyết định `min`/`max` theo chỉ số **cũ** `2*i` thay vì `i`

`2*i` luôn chẵn nên lúc nào cũng chọn `min`. Phải xét tính chẵn/lẻ của chỉ số **mới** `i`.

In [20]:
def sai_chi_so_cu(nums):
    nums = list(nums); n = len(nums)
    while n > 1:
        n //= 2
        for i in range(n):
            a, b = nums[2*i], nums[2*i+1]
            nums[i] = min(a, b) if (2 * i) % 2 == 0 else max(a, b)  # SAI: luôn là min
    return nums[0]

print("sai :", sai_chi_so_cu([4, 3, 2, 1]), "| đúng:", SolutionInPlace().minMaxGame([4, 3, 2, 1]))

sai : 1 | đúng: 2


### Lỗi 3: Ghi đè từ cuối về đầu

Nếu duyệt `i` giảm dần (`n-1, ..., 0`) thì ghi `nums[i]` có thể đè lên ô `2i'` mà vòng sau cần. Phải đi `i` **tăng dần**.

In [21]:
def sai_nguoc(nums):
    nums = list(nums); n = len(nums)
    while n > 1:
        n //= 2
        for i in range(n - 1, -1, -1):                              # SAI: duyệt ngược
            a, b = nums[2*i], nums[2*i+1]
            nums[i] = min(a, b) if i % 2 == 0 else max(a, b)
    return nums[0]

arr = [1, 3, 5, 2, 4, 8, 2, 2]
print("sai :", sai_nguoc(arr), "| đúng:", SolutionInPlace().minMaxGame(arr))
arr2 = [5, 1, 2, 8, 3, 3, 9, 4]
print("sai :", sai_nguoc(arr2), "| đúng:", SolutionInPlace().minMaxGame(arr2))

sai : 1 | đúng: 1
sai : 5 | đúng: 1


### Lỗi 4: Dùng `n / 2` (chia thực) thay vì `n // 2`

`n / 2` trả về số thực (`4.0`) nên `range(4.0)` báo `TypeError`; luôn dùng chia nguyên `//` cho độ dài/chỉ số.

In [22]:
try:
    for i in range(8 / 2):
        pass
except TypeError as e:
    print("TypeError:", e)
print(8 // 2, type(8 // 2), type(8 / 2))

TypeError: 'float' object cannot be interpreted as an integer
4 <class 'int'> <class 'float'>


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề tả một thuật toán từng bước, `n <= 1024`: đây là bài **mô phỏng**, cứ làm đúng đề.
- Mỗi vòng chia đôi độ dài -> tổng `O(n)`; nhận ra chuỗi hình học `n + n/2 + ...`.
- Chẵn/lẻ theo chỉ số mới: `i % 2`; cặp liền kề: `2*i, 2*i+1`.
- Muốn tiết kiệm bộ nhớ: ghi đè in-place **từ trái sang phải**.
- Cấu trúc "ghép cặp liền kề theo tầng" thường nhắc đến **cây nhị phân/tournament**; bài tương tự: LeetCode 2169 (Count Operations to Obtain Zero) khác dạng, nhưng cùng tư duy mô phỏng.

## 15. Biến thể của bài

- **Độ dài không phải lũy thừa của 2:** phần tử lẻ thừa xử lý thế nào (giữ nguyên, bỏ, hoặc đẩy sang tầng sau).
- **Luân phiên theo tầng** thay vì theo chỉ số: tầng 1 `min`, tầng 2 `max`, ... (Tournament).
- **Trả về mảng ở mọi tầng**, hoặc trả về số vòng.
- **Cập nhật động:** đổi một phần tử và tính lại đáp án nhanh (cần **segment tree** với luân phiên min/max).
- **Nhiều phép toán hơn:** cộng/nhân luân phiên, tương tự bài "Minimum Operations".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng mới | làm đúng đề | `O(n)` | `O(n)` | dễ đúng nhất, người mới |
| 2. In-place | ghi đè trái -> phải | `O(n)` | `O(1)` | submit, tiết kiệm bộ nhớ |
| 3. Đệ quy | `play(half)` | `O(n)` | `O(n)` | tư duy cây/chia nhỏ |
| 4. `zip` | cặp liền kề bằng slicing | `O(n)` | `O(n)` | code ngắn, đọc như đề |
| 5. Stride | khối `2*step`, ghi ô đầu | `O(n)` | `O(1)` | hiểu cấu trúc cây nhị phân |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().minMaxGame([6, 1, 2, 9])` (nhớ: chỉ số mới `0` là min, `1` là max).

**Bài 2 (tính tay).** Với `[8, 2, 4, 7, 1, 6, 5, 3]`, viết ra mảng sau mỗi vòng và đáp án.

**Bài 3 (điền TODO).** Hoàn thành hàm `one_round(nums)` trả về mảng sau đúng **một** vòng (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `rounds_count(n)` trả về số vòng cho độ dài `n` là lũy thừa của 2 (không dùng `math`).

**Bài 5 (mini challenge).** Viết `all_levels(nums)` trả về **danh sách tất cả các mảng** từ mảng đầu tiên đến mảng một phần tử.

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def one_round_todo(nums):
    half = []
    # for i in range(len(nums) // 2):
    #     ...
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(6,1)` ở vị trí 0 -> min `1`; `(2,9)` ở vị trí 1 -> max `9`; rồi `(1,9)` ở vị trí 0 -> min `1`. Bài 2: viết từng vòng một.
- **Mức 2 (vừa):** Bài 3: dùng `min` khi `i % 2 == 0`, ngược lại `max`. Bài 4: đếm `while n > 1: n //= 2; c += 1`.
- **Mức 3 (sâu):** Bài 5: lặp `while len(cur) > 1`, mỗi vòng `levels.append(cur)`; nhớ thêm mảng cuối cùng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1 và 2
assert Solution().minMaxGame([6, 1, 2, 9]) == 1
assert Solution().minMaxGame([8, 2, 4, 7, 1, 6, 5, 3]) == 2

# Bài 3
def one_round(nums):
    return [min(nums[2 * i], nums[2 * i + 1]) if i % 2 == 0 else max(nums[2 * i], nums[2 * i + 1])
            for i in range(len(nums) // 2)]

# Bài 4
def rounds_count(n):
    c = 0
    while n > 1:
        n //= 2
        c += 1
    return c

# Bài 5
def all_levels(nums):
    levels = [list(nums)]
    while len(levels[-1]) > 1:
        levels.append(one_round(levels[-1]))
    return levels

assert one_round([8, 2, 4, 7, 1, 6, 5, 3]) == [2, 7, 1, 5]
assert one_round([2, 7, 1, 5]) == [2, 5]
assert rounds_count(1) == 0 and rounds_count(2) == 1 and rounds_count(1024) == 10
assert all_levels([1, 3, 5, 2, 4, 8, 2, 2]) == [[1, 3, 5, 2, 4, 8, 2, 2], [1, 5, 4, 2], [1, 4], [1]]
assert all_levels([3]) == [[3]]
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ghi đè tại chỗ, trái -> phải)

```python
class Solution:
    def minMaxGame(self, nums: List[int]) -> int:
        n = len(nums)
        while n > 1:
            n //= 2
            for i in range(n):
                a, b = nums[2 * i], nums[2 * i + 1]
                nums[i] = min(a, b) if i % 2 == 0 else max(a, b)
        return nums[0]
```

**Checklist**

- [ ] Chỉ số mới chẵn dùng `min`, lẻ dùng `max` (không đảo)?
- [ ] Quyết định theo `i` (chỉ số mới), không theo `2*i`?
- [ ] Duyệt `i` tăng dần khi ghi đè in-place?
- [ ] Dùng `//` thay vì `/` cho độ dài?
- [ ] Trường hợp `n == 1` trả thẳng `nums[0]`?
- [ ] Signature đúng `minMaxGame(self, nums: List[int]) -> int` và có `from typing import List`?